# Mutation Testing Demo on Fastapi Repository

Repository from: https://github.com/fastapi/fastapi

## Project Exploration

In [1]:
from pymut4se.api import discover

workspace = discover("../fastapi-master") # Root of fastapi

print(workspace.project.name)
print(workspace.statistics())
print(workspace.statistics().as_dict())

fastapi-master
ProjectStatistics(packages=95, modules=431, chunks=1128, tests=2259, tested_chunks=96, requirements=62)
{'packages': 95, 'modules': 431, 'original_chunks': 1128, 'test_suites': 704, 'test_cases': 2259, 'test_links': 2528, 'chunks_with_tests': 96, 'requirements': 62}


In [2]:
modules = workspace.find_modules()
chunks = workspace.find_chunks()
print(modules)


[Module(name='docs_src.additional_responses.tutorial001_py310', path='docs_src/additional_responses/tutorial001_py3…', id='134b87fa…'), Module(name='docs_src.additional_responses.tutorial002_py310', path='docs_src/additional_responses/tutorial002_py3…', id='655f0d75…'), Module(name='docs_src.additional_responses.tutorial003_py310', path='docs_src/additional_responses/tutorial003_py3…', id='ea1c96ed…'), Module(name='docs_src.additional_responses.tutorial004_py310', path='docs_src/additional_responses/tutorial004_py3…', id='d88546ec…'), Module(name='docs_src.additional_status_codes.tutorial001_…', path='docs_src/additional_status_codes/tutorial001_…', id='8d08dafe…'), Module(name='docs_src.additional_status_codes.tutorial001_…', path='docs_src/additional_status_codes/tutorial001_…', id='afb1b317…'), Module(name='docs_src.advanced_middleware.tutorial001_py310', path='docs_src/advanced_middleware/tutorial001_py31…', id='e7ea4ce7…'), Module(name='docs_src.advanced_middleware.tutorial002_py3

### Available mutation operators

In [3]:
for operator in workspace.operators():
    print(operator.name, operator.class_name, operator.description)

add-if-not-null IfNotNullMutation Guard a function body so it only runs for non-null inputs.
arithmetic ArithmeticMutation Replace one arithmetic binary operator.
boolean-replacement BooleanReplacementMutation Invert one boolean literal.
constant-replacement ConstantReplacementMutation Transform one numeric constant.
control-replacement ControlReplacementMutation Swap break and continue.
delete-assignment DeleteAssignmentMutation Delete one assignment statement.
delete-decorator DeleteDecoratorMutation Delete one function decorator.
delete-if-statement DeleteIfStatementMutation Delete one complete if statement.
delete-return DeleteReturnMutation Delete one return statement.
delete-while DeleteWhileMutation Delete one complete while loop.
logical-connector LogicalConnectorMutation Swap and and or in one boolean expression.
optional-parameter-callee OptionalParamCalleeMutation Change an optional parameter default in the called function.
optional-parameter-caller OptionalParamCallerMutati

## Mutant Generation

In [4]:
""" 
In this demo we will generate mutants for all the code chunks that are found in the fastapi source code.
By default workspace.mutate() will apply all the operators and generate mutants of degree 1.
You can set the operators to any of the available operators.
Note: increasing the max_degree will quickly result in a very large number of mutants if you target an entire project. 
To speed un the generation, you can alo chose to only generate mutants for code chunks with test cases by using:
new_mutants = workspace.mutate_chunks_with_tests(workspace.chunks, "all")
"""

targets = [
    *workspace.find_modules("fastapi"), # you can change the module if you prefer testing only one specific module.
    *workspace.find_chunks("_apply_mutation"),
]

new_mutants = workspace.mutate(
    targets,
    operators="all",
    max_degree=1,
)


Generating mutants: 7 new | chunks processed: 0/351

Generating mutants: 26854 new | chunks processed: 351/351


In [5]:
"""
You can check the number of mutants per mutation operator.
"""
print(workspace.mutant_statistics())

MutantStatistics(total=26854, source_chunks=351, by_degree={1: 26854}, by_type={'arithmetic': 12045, 'boolean_replacement': 314, 'cast_type': 4812, 'constant_replacement': 352, 'control_replacement': 15, 'delete_assignment': 1080, 'delete_decorator': 33, 'delete_if_statement': 566, 'delete_return': 458, 'delete_while': 3, 'if_not_null': 1604, 'logical_connector': 223, 'optional_param_callee': 1138, 'optional_param_caller': 171, 'relational': 2880, 'return_pass': 300, 'swap_arguments': 605, 'unary': 255}, input_executions=0, test_executions=0, failed_tests=0)


## Mutant execution with test suite

In [6]:
"""
Before starting the execution it is worth checking for which mutants there exist test cases.
If no test cases are found for a code chunk, by default the whole test-suite will be executed.
For better performance you can chose to only execute the mutants with test cases.
"""

mutants_with_tests = [
    mutant for mutant in new_mutants
    if mutant.related_test_cases
]
len(mutants_with_tests)

4339

In [7]:
test_outputs = workspace.run_tests(
    mutants_with_tests,
    parallel=True,
    max_workers=6,
    timeout_seconds=2,
)

Preparing execution environment...
Execution environment ready: /Users/laura/Desktop/exp_tool/PyMut4SE/fastapi-master/.pymut4se/venvs/16c4f03101df6faf3f30707fef5ce139bee9803ccecfb758accd08cd3ab3d75d
Executing tests: 4339/4339


In [8]:
"""
Print the mutation score and the amount of killed and survived mutants.
"""

score = workspace.mutation_score()
print(score)

MutationScore(score=33.69%, killed=892, survived=1756, untested=22515, incomplete=0, errors=1691)


## Storing mutants & execution in database

In [9]:
from sqlalchemy import create_engine
from sqlalchemy.orm import Session

from pymut4se.model import Base

engine = create_engine("sqlite:///fastapi.db")
Base.metadata.create_all(engine)

with Session(engine) as session:
    workspace.save(session, commit=True)